In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ===================== USER SETTINGS =====================
MODE = "single"      
# Options:
# "single" -> one CSV file
# "batch"  -> all CSV files in a folder

SINGLE_FILE_PATH = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/drill_mech_depth_30jul20.csv"
SINGLE_DESC_PATH = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/desc/drill_mech_depth_30jul20_curve_summary.txt"

BATCH_DESC_FOLDER = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/desc/"
BATCH_FOLDER_PATH = "/mnt/data/"   # change if needed

OUTPUT_FOLDER = "/Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# ==========================================================


DUMMY_VALUE = -999.25

# --------- MANUAL SCALE CONTROL ----------
MANUAL_X_SCALE = {
    # Example:
    # "GR": (0, 150),
    # "RHOB": (1.9, 2.9),
}
# -----------------------------------------


def get_auto_scale(series):
    clean = series.dropna()
    if len(clean) == 0:
        return (0, 1)
    vmin = np.percentile(clean, 2)
    vmax = np.percentile(clean, 98)
    return float(vmin), float(vmax)

def load_curve_descriptions(desc_file_path):
    """
    Reads *_curve_summary.txt and returns:
    {MNEM: (UNIT, DESCRIPTION)}
    """
    descriptions = {}
    if not os.path.exists(desc_file_path):
        return descriptions

    with open(desc_file_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        if line.startswith("~") or line.startswith("#") or len(line.strip()) == 0:
            continue

        parts = line.strip().split(None, 2)
        if len(parts) < 3:
            continue

        mnem = parts[0]
        unit = parts[1].replace(".", "")
        desc = parts[2]

        descriptions[mnem] = (unit, desc)

    return descriptions

def plot_well_log(csv_path, save_folder, desc_file_path=None):

    # ---- Load descriptions for this file ----
    CURVE_DESCRIPTIONS = load_curve_descriptions(desc_file_path)

    df = pd.read_csv(csv_path)
    df = df.apply(pd.to_numeric, errors='coerce')

    df.replace(DUMMY_VALUE, np.nan, inplace=True)
    df.dropna(how="all", inplace=True)

    y = df.iloc[:, 0]
    features = df.columns[1:]
    n_tracks = len(features)

    fig, axes = plt.subplots(
        nrows=1, ncols=n_tracks,
        figsize=(3*n_tracks, 14),
        sharey=True
    )

    if n_tracks == 1:
        axes = [axes]

    scale_report = {}

    for i, feature in enumerate(features):
        ax = axes[i]
        curve = df[feature]

        ax.plot(curve, y, linewidth=0.8)

        # ---- Scaling ----
        if feature in MANUAL_X_SCALE:
            xmin, xmax = MANUAL_X_SCALE[feature]
            scale_type = "MANUAL"
        else:
            xmin, xmax = get_auto_scale(curve)
            scale_type = "AUTO"

        ax.set_xlim(xmin, xmax)
        ax.margins(x=0)
        ax.set_xlabel(feature)
        ax.grid(True)
        ax.invert_yaxis()

        scale_report[feature] = (scale_type, xmin, xmax)

        # ---- Add description under track ----
        if feature in CURVE_DESCRIPTIONS:
            unit, desc = CURVE_DESCRIPTIONS[feature]
            text = f"{feature} ({unit})\n{desc}"
        else:
            text = f"{feature}\n(No description available)"

        ax.text(
            0.5, -0.13, text,
            ha="center", va="top",
            transform=ax.transAxes,
            fontsize=8,
            wrap=True
        )

    axes[0].set_ylabel(df.columns[0])
    plt.suptitle(os.path.basename(csv_path), fontsize=14)

    os.makedirs(save_folder, exist_ok=True)
    out_name = os.path.splitext(os.path.basename(csv_path))[0] + "_welllog.png"
    out_path = os.path.join(save_folder, out_name)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    # ---- Print scale report ----
    print("\n--- X-AXIS SCALE REPORT ---")
    for feat, info in scale_report.items():
        print(f"{feat:15s} | {info[0]:6s} | xmin={info[1]:.3f} , xmax={info[2]:.3f}")
    print("----------------------------")
    print("Saved plot:", out_path)

# ===================== RUN =====================
if MODE == "single":
    plot_well_log(
        SINGLE_FILE_PATH,
        OUTPUT_FOLDER,
        desc_file_path=SINGLE_DESC_PATH
    )

elif MODE == "batch":
    for file in os.listdir(BATCH_FOLDER_PATH):
        if file.lower().endswith(".csv"):
            csv_path = os.path.join(BATCH_FOLDER_PATH, file)

            # matching description file
            base = os.path.splitext(file)[0]
            desc_path = os.path.join(BATCH_DESC_FOLDER, base + "_curve_summary.txt")

            plot_well_log(
                csv_path,
                OUTPUT_FOLDER,
                desc_file_path=desc_path
            )

print("All done.")


/var/folders/7x/n43ldjyn17n1mx3b563hl8r00000gn/T/ipykernel_63098/3656200974.py:110: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(xmin, xmax)



--- X-AXIS SCALE REPORT ---
HKLA            | AUTO   | xmin=1.020 , xmax=1.020
SWOB            | AUTO   | xmin=11.871 , xmax=18.172
RPM             | AUTO   | xmin=0.000 , xmax=44.690
TFLO            | AUTO   | xmin=1199.694 , xmax=1711.912
SPPA            | AUTO   | xmin=4955.619 , xmax=11038.140
ROP5            | AUTO   | xmin=3.025 , xmax=200.348
BLKP            | AUTO   | xmin=1.173 , xmax=14.188
STOR            | AUTO   | xmin=0.000 , xmax=8.087
TRPM            | AUTO   | xmin=2003.000 , xmax=3297.610
TEMP_DNI        | AUTO   | xmin=39.217 , xmax=52.550
GR              | AUTO   | xmin=8.690 , xmax=44.284
CRPM            | AUTO   | xmin=0.000 , xmax=44.348
STICKNSLIP      | AUTO   | xmin=0.000 , xmax=117.168
----------------------------
Saved plot: /Users/apple/Downloads/welldata/ROTTERDAM-08-SIDETRACK1/structured/csv/drill_mech_depth_30jul20_welllog.png
All done.
